In [ ]:
import requests
import pandas as pd
import numpy as np
import sys
import os
from os.path import join
import math
import time
import logging
import matplotlib.pyplot as plt
from tensorflow.keras import backend as K
import xgboost as xgb
from sklearn.metrics import r2_score

from rdkit import Chem
from rdkit.Chem import Crippen
from rdkit.Chem import Descriptors

logfile = "datacount_sabio.log"

from directory_infomation import *

from build_GNN import *
from functions_and_dicts_data_preprocessing_GNN import *
from functions_for_data_preprocessing import *
#from functions_for_unirep_calculations import *
sabio_df = pd.read_pickle('KM_new.pkl')
sabio_df["MW"] = np.nan
sabio_df["LogP"] = np.nan

for ind in sabio_df.index:
    kegg_id = sabio_df["KEGG ID"][ind]
    try:
        mol = Chem.MolFromMolFile(datasets_dir +  "/mol-files/" + kegg_id + '.mol')
    except OSError:
        None
    if mol is not None:
        sabio_df["MW"][ind] = Descriptors.ExactMolWt(mol)
        sabio_df["LogP"][ind] = Crippen.MolLogP(mol)
droplist = []
for ind in sabio_df.index:
    droplist.append(calculate_and_save_input_matrixes(sample_ID = "sabio_" + str(ind), 
                                                      molecule_ID = sabio_df["KEGG ID"][ind],
                                                      unirep = sabio_df["Unirep"][ind], 
                                                      extras = np.array([sabio_df["MW"][ind], sabio_df["LogP"][ind]]),
                                                      save_folder = join(datasets_dir, "GNN_input_data")))
droplist = list(np.array(droplist)[np.array(droplist) != -1])
sabio_df.drop(droplist, inplace = True)
# sabio_df.to_pickle("with_GNN_data.pkl")

2024-12-10 14:18:03.292731: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcudart.so.10.1'; dlerror: libcudart.so.10.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/include/nccl.h
2024-12-10 14:18:03.292759: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
/home/wuke/anaconda3/envs/km_pre/lib/python3.7/site-packages/sklearn/base.py:338: UserWarning: Trying to unpickle estimator PCA from version 0.23.1 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  UserWarning,
/home/wuke/anaconda3/envs/km_pre/lib/python3.7/site-packages/ipykernel_launcher.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

Could not create input for substrate ID C13183
Could not create input for substrate ID C16528
Could not create input for substrate ID C16097
Could not create input for substrate ID C13183
Could not create input for substrate ID D03336
Could not create input for substrate ID C02135
Could not create input for substrate ID C01836
Could not create input for substrate ID C01493
Could not create input for substrate ID C00138
Could not create input for substrate ID C00996
Could not create input for substrate ID D03336
Could not create input for substrate ID C00541
Could not create input for substrate ID C00807
Could not create input for substrate ID C16098
Could not create input for substrate ID C08908
Could not create input for substrate ID C00541
Could not create input for substrate ID C13452
Could not create input for substrate ID C00853
Could not create input for substrate ID C00139
Could not create input for substrate ID C13183
Could not create input for substrate ID C21471
Could not cre

In [3]:
batch_size = 64
D = 50
learning_rate = 0.05
epochs = 50
l2_reg_fc = 1
l2_reg_conv = 0.01
rho = 0.95


model = DMPNN(l2_reg_conv = l2_reg_conv, l2_reg_fc = l2_reg_fc, learning_rate = learning_rate,
                  D = D, N = N, F1 = F1, F2 = F2, F= F, drop_rate = 0.0, ada_rho = rho)
model.load_weights(join(datasets_dir, "model_weights", "saved_model_GNN_best_hyperparameters"))

get_fingerprint_fct = K.function([model.layers[0].input, model.layers[26].input,  model.layers[3].input, model.layers[36].input],
                                  [model.layers[-10].output])
def get_substrate_representations(df, prefix):
    df["GNN FP"] = ""
    i = 0
    n = len(df)
    UniRep = np.zeros((64, 20))
    
    cid_all = list(df.index)
    cid_all = [prefix + str(cid) for cid in cid_all]
    
    while i*64 <= n:
        if (i+1)*64  <= n:
            XE, X, A, extras = get_representation_input(cid_all[i*64:(i+1)*64])
            representations = get_fingerprint_fct([np.array(XE), np.array(X),np.array(A),
                                                   np.array(extras)])[0]
            df["GNN FP"][i*64:(i+1)*64] = list(representations[:, :52])
        else:
            print(i)
            XE, X, A, extras = get_representation_input(cid_all[-64:])
            representations = get_fingerprint_fct([np.array(XE), np.array(X),np.array(A), 
                                                   np.array(extras)])[0]
            df["GNN FP"][-64:] = list(representations[:, :52])
        i += 1
        
    ### set all GNN FP-entries with no input matrices to np.nan:
    all_X_matrices = os.listdir(input_data_folder)
    for ind in df.index:
        if prefix +str(ind) +"_X.npy" not in all_X_matrices:
            df["GNN FP"][ind] = np.nan
    return(df)
sabio_df = get_substrate_representations(df = sabio_df, prefix = "sabio_")

2024-12-10 14:27:58.716201: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcuda.so.1
2024-12-10 14:27:58.801552: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1716] Found device 0 with properties: 
pciBusID: 0000:39:00.0 name: NVIDIA A800 80GB PCIe computeCapability: 8.0
coreClock: 1.41GHz coreCount: 108 deviceMemorySize: 79.25GiB deviceMemoryBandwidth: 1.76TiB/s
2024-12-10 14:27:58.802656: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1716] Found device 1 with properties: 
pciBusID: 0000:9c:00.0 name: NVIDIA A800 80GB PCIe computeCapability: 8.0
coreClock: 1.41GHz coreCount: 108 deviceMemorySize: 79.25GiB deviceMemoryBandwidth: 1.76TiB/s
2024-12-10 14:27:58.803676: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1716] Found device 2 with properties: 
pciBusID: 0000:9d:00.0 name: NVIDIA A800 80GB PCIe computeCapability: 8.0
coreClock: 1.41GHz coreCount: 108 deviceMemorySize: 79.25GiB deviceMemoryBandwidth: 1.76TiB/s
2

322


/home/wuke/anaconda3/envs/km_pre/lib/python3.7/site-packages/ipykernel_launcher.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [5]:
sabio_df.to_pickle("with_GNN_data.pkl")